# EDF to NPY

In [1]:
import os

edf_root = r"D:\V89\data2\EDF"  # เปลี่ยนเป็นโฟลเดอร์ EDF ของคุณ

edf_files = [f for f in os.listdir(edf_root) if f.lower().endswith('.edf')]

print(f"📂 Found {len(edf_files)} EDF files in: {edf_root}")
for i, f in enumerate(edf_files[:10]):  # โชว์ตัวอย่าง 10 ไฟล์แรก
    size_mb = os.path.getsize(os.path.join(edf_root, f)) / (1024*1024)
    print(f"{i+1:02d}. {f} - {size_mb:.2f} MB")

if len(edf_files) > 10:
    print("... (showing first 10 only)")

📂 Found 78 EDF files in: D:\V89\data2\EDF
01. 00000995-100507[001].edf - 677.81 MB
02. 00000995-100507[002].edf - 677.81 MB
03. 00000995-100507[003].edf - 677.81 MB
04. 00000995-100507[004].edf - 677.81 MB
05. 00000995-100507[005].edf - 657.29 MB
06. 00000999-100507[001].edf - 677.81 MB
07. 00000999-100507[002].edf - 677.81 MB
08. 00000999-100507[003].edf - 677.81 MB
09. 00000999-100507[004].edf - 677.81 MB
10. 00000999-100507[005].edf - 94.33 MB
... (showing first 10 only)


## Test

In [ ]:
import os
import numpy as np
import librosa
import pyedflib
import time
from pathlib import Path

# ========================
# Configuration
# ========================
EDF_DIR = r"D:\V89\data_test\EDF"       # โฟลเดอร์ EDF
NPY_DIR = r"C:\V89\data_test\test_npy" # โฟลเดอร์เก็บ NPY
TARGET_SR = 16000                   # Sampling rate เป้าหมาย
CHANNELS = ["Mic"]                  # ใช้แค่ Mic channel

os.makedirs(NPY_DIR, exist_ok=True)

# ========================
# Find EDF files
# ========================
edf_files = [f for f in os.listdir(EDF_DIR) if f.lower().endswith(".edf")]
print(f"📁 Found {len(edf_files)} EDF files")

if len(edf_files) == 0:
    print("❌ No EDF files found!")
    exit()

# ========================
# Conversion with timing
# ========================
start_time = time.time()
successful = 0
failed = 0

for i, edf_file in enumerate(edf_files, 1):
    edf_path = os.path.join(EDF_DIR, edf_file)
    npy_path = os.path.join(NPY_DIR, edf_file.replace(".edf", "_mic.npy"))
    
    print(f"\n[{i}/{len(edf_files)}] Processing: {edf_file}")
    
    try:
        file_start = time.time()
        
        with pyedflib.EdfReader(edf_path) as f:
            sig_labels = f.getSignalLabels()
            print(f"   📋 Available channels: {sig_labels}")
            
            # หา Mic channel (case insensitive)
            mic_idx = None
            for idx, label in enumerate(sig_labels):
                if "mic" in label.lower():
                    mic_idx = idx
                    actual_ch_name = label
                    break
            
            if mic_idx is None:
                print(f"   ⚠️ No Mic channel found, skipping...")
                failed += 1
                continue
            
            print(f"   🎤 Using channel: '{actual_ch_name}' (index: {mic_idx})")
            
            # Read signal
            signal = f.readSignal(mic_idx).astype(np.float32)
            original_sr = f.getSampleFrequency(mic_idx)
            duration_sec = len(signal) / original_sr
            
            print(f"   📊 Original: {original_sr}Hz, {len(signal):,} samples, {duration_sec:.1f}s")
            
            # Resample if needed
            if original_sr != TARGET_SR:
                print(f"   🔄 Resampling: {original_sr}Hz → {TARGET_SR}Hz")
                signal = librosa.resample(signal, orig_sr=original_sr, target_sr=TARGET_SR)
                print(f"   ✅ New length: {len(signal):,} samples")
        
        # Save as NPY
        np.save(npy_path, signal)
        file_time = time.time() - file_start
        file_size_mb = os.path.getsize(npy_path) / (1024*1024)
        
        print(f"   💾 Saved: {Path(npy_path).name}")
        print(f"   ⏱️ Processing time: {file_time:.2f}s")
        print(f"   📦 File size: {file_size_mb:.1f} MB")
        successful += 1
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        failed += 1

# ========================
# Summary
# ========================
total_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"🎯 CONVERSION COMPLETED!")
print(f"✅ Successful: {successful} files")
print(f"❌ Failed: {failed} files")
print(f"⏱️ Total time: {total_time:.2f}s")
print(f"⚡ Average time per file: {total_time/len(edf_files):.2f}s")
print(f"📁 Output directory: {NPY_DIR}")

# ========================
# Performance Analysis
# ========================
print(f"\n{'='*50}")
print(f"📈 PERFORMANCE BENEFITS:")
print(f"")
print(f"🔄 Training Speed:")
print(f"   • EDF loading: ~1-5s per file (complex format)")
print(f"   • NPY loading: ~0.01-0.1s per file (binary format)")
print(f"   • Speed up: 10-50x faster! 🚀")
print(f"")
print(f"💾 Memory Efficiency:")
print(f"   • EDF: Header parsing + channel selection")
print(f"   • NPY: Direct memory mapping")
print(f"   • Less RAM usage during training")
print(f"")
print(f"🎯 Training Benefits:")
print(f"   • Faster data loading = shorter epoch time")
print(f"   • Consistent sample rate ({TARGET_SR}Hz)")
print(f"   • No runtime resampling needed")
print(f"   • Better GPU utilization (less I/O wait)")

# ========================
# Usage Example
# ========================
print(f"\n{'='*50}")
print(f"💡 HOW TO LOAD NPY FILES:")
print(f"")
print(f"# Single file")
print(f"mic_data = np.load('path/to/file_mic.npy')")
print(f"print(f'Shape: {{mic_data.shape}}, SR: {TARGET_SR}Hz')")
print(f"")
print(f"# Batch loading for training")
print(f"import glob")
print(f"npy_files = glob.glob('{NPY_DIR}/*_mic.npy')")
print(f"for npy_file in npy_files:")
print(f"    data = np.load(npy_file)  # Very fast!")
print(f"    # Your training code here...")

📁 Found 15 EDF files

[1/15] Processing: 00001010-100507[001].edf
   📋 Available channels: ['EEG A1-A2', 'EEG C3-A2', 'EEG C4-A1', 'EOG LOC-A2', 'EOG ROC-A2', 'EMG Chin', 'Leg 1', 'Leg 2', 'ECG I', 'RR', 'Snore', 'Flow Patient', 'Flow Patient', 'Effort THO', 'Effort ABD', 'SpO2', 'Body', 'PulseRate', 'Mic', 'Tracheal']
   🎤 Using channel: 'Mic' (index: 18)
   📊 Original: 48000.0Hz, 172,800,000 samples, 3600.0s
   🔄 Resampling: 48000.0Hz → 16000Hz
   ✅ New length: 57,600,000 samples
   💾 Saved: 00001010-100507[001]_mic.npy
   ⏱️ Processing time: 15.66s
   📦 File size: 219.7 MB

[2/15] Processing: 00001010-100507[002].edf
   📋 Available channels: ['EEG A1-A2', 'EEG C3-A2', 'EEG C4-A1', 'EOG LOC-A2', 'EOG ROC-A2', 'EMG Chin', 'Leg 1', 'Leg 2', 'ECG I', 'RR', 'Snore', 'Flow Patient', 'Flow Patient', 'Effort THO', 'Effort ABD', 'SpO2', 'Body', 'PulseRate', 'Mic', 'Tracheal']
   🎤 Using channel: 'Mic' (index: 18)
   📊 Original: 48000.0Hz, 172,800,000 samples, 3600.0s
   🔄 Resampling: 48000.0

## Train

In [5]:
import os
import numpy as np
import librosa
import pyedflib
import time
from pathlib import Path

# ========================
# Configuration
# ========================
EDF_DIR = r"D:\V89\data2\EDF"       # โฟลเดอร์ EDF
NPY_DIR = r"D:\V89\data2\train_npy" # โฟลเดอร์เก็บ NPY
TARGET_SR = 16000                   # Sampling rate เป้าหมาย
CHANNELS = ["Mic"]                  # ใช้แค่ Mic channel

os.makedirs(NPY_DIR, exist_ok=True)

# ========================
# Find EDF files
# ========================
edf_files = [f for f in os.listdir(EDF_DIR) if f.lower().endswith(".edf")]
print(f"📁 Found {len(edf_files)} EDF files")

if len(edf_files) == 0:
    print("❌ No EDF files found!")
    exit()

# ========================
# Conversion with timing
# ========================
start_time = time.time()
successful = 0
failed = 0

for i, edf_file in enumerate(edf_files, 1):
    edf_path = os.path.join(EDF_DIR, edf_file)
    npy_path = os.path.join(NPY_DIR, edf_file.replace(".edf", "_mic.npy"))
    
    print(f"\n[{i}/{len(edf_files)}] Processing: {edf_file}")
    
    try:
        file_start = time.time()
        
        with pyedflib.EdfReader(edf_path) as f:
            sig_labels = f.getSignalLabels()
            print(f"   📋 Available channels: {sig_labels}")
            
            # หา Mic channel (case insensitive)
            mic_idx = None
            for idx, label in enumerate(sig_labels):
                if "mic" in label.lower():
                    mic_idx = idx
                    actual_ch_name = label
                    break
            
            if mic_idx is None:
                print(f"   ⚠️ No Mic channel found, skipping...")
                failed += 1
                continue
            
            print(f"   🎤 Using channel: '{actual_ch_name}' (index: {mic_idx})")
            
            # Read signal
            signal = f.readSignal(mic_idx).astype(np.float32)
            original_sr = f.getSampleFrequency(mic_idx)
            duration_sec = len(signal) / original_sr
            
            print(f"   📊 Original: {original_sr}Hz, {len(signal):,} samples, {duration_sec:.1f}s")
            
            # Resample if needed
            if original_sr != TARGET_SR:
                print(f"   🔄 Resampling: {original_sr}Hz → {TARGET_SR}Hz")
                signal = librosa.resample(signal, orig_sr=original_sr, target_sr=TARGET_SR)
                print(f"   ✅ New length: {len(signal):,} samples")
        
        # Save as NPY
        np.save(npy_path, signal)
        file_time = time.time() - file_start
        file_size_mb = os.path.getsize(npy_path) / (1024*1024)
        
        print(f"   💾 Saved: {Path(npy_path).name}")
        print(f"   ⏱️ Processing time: {file_time:.2f}s")
        print(f"   📦 File size: {file_size_mb:.1f} MB")
        successful += 1
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        failed += 1

# ========================
# Summary
# ========================
total_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"🎯 CONVERSION COMPLETED!")
print(f"✅ Successful: {successful} files")
print(f"❌ Failed: {failed} files")
print(f"⏱️ Total time: {total_time:.2f}s")
print(f"⚡ Average time per file: {total_time/len(edf_files):.2f}s")
print(f"📁 Output directory: {NPY_DIR}")

# ========================
# Performance Analysis
# ========================
print(f"\n{'='*50}")
print(f"📈 PERFORMANCE BENEFITS:")
print(f"")
print(f"🔄 Training Speed:")
print(f"   • EDF loading: ~1-5s per file (complex format)")
print(f"   • NPY loading: ~0.01-0.1s per file (binary format)")
print(f"   • Speed up: 10-50x faster! 🚀")
print(f"")
print(f"💾 Memory Efficiency:")
print(f"   • EDF: Header parsing + channel selection")
print(f"   • NPY: Direct memory mapping")
print(f"   • Less RAM usage during training")
print(f"")
print(f"🎯 Training Benefits:")
print(f"   • Faster data loading = shorter epoch time")
print(f"   • Consistent sample rate ({TARGET_SR}Hz)")
print(f"   • No runtime resampling needed")
print(f"   • Better GPU utilization (less I/O wait)")

# ========================
# Usage Example
# ========================
print(f"\n{'='*50}")
print(f"💡 HOW TO LOAD NPY FILES:")
print(f"")
print(f"# Single file")
print(f"mic_data = np.load('path/to/file_mic.npy')")
print(f"print(f'Shape: {{mic_data.shape}}, SR: {TARGET_SR}Hz')")
print(f"")
print(f"# Batch loading for training")
print(f"import glob")
print(f"npy_files = glob.glob('{NPY_DIR}/*_mic.npy')")
print(f"for npy_file in npy_files:")
print(f"    data = np.load(npy_file)  # Very fast!")
print(f"    # Your training code here...")

📁 Found 78 EDF files

[1/78] Processing: 00000995-100507[001].edf
   📋 Available channels: ['EEG A1-A2', 'EEG C3-A2', 'EEG C4-A1', 'EOG LOC-A2', 'EOG ROC-A2', 'EMG Chin', 'Leg 1', 'Leg 2', 'ECG I', 'RR', 'Snore', 'Flow Patient', 'Flow Patient', 'Effort THO', 'Effort ABD', 'SpO2', 'Body', 'PulseRate', 'Mic', 'Tracheal']
   🎤 Using channel: 'Mic' (index: 18)
   📊 Original: 48000.0Hz, 172,800,000 samples, 3600.0s
   🔄 Resampling: 48000.0Hz → 16000Hz
   ✅ New length: 57,600,000 samples
   💾 Saved: 00000995-100507[001]_mic.npy
   ⏱️ Processing time: 26.61s
   📦 File size: 219.7 MB

[2/78] Processing: 00000995-100507[002].edf
   📋 Available channels: ['EEG A1-A2', 'EEG C3-A2', 'EEG C4-A1', 'EOG LOC-A2', 'EOG ROC-A2', 'EMG Chin', 'Leg 1', 'Leg 2', 'ECG I', 'RR', 'Snore', 'Flow Patient', 'Flow Patient', 'Effort THO', 'Effort ABD', 'SpO2', 'Body', 'PulseRate', 'Mic', 'Tracheal']
   🎤 Using channel: 'Mic' (index: 18)
   📊 Original: 48000.0Hz, 172,800,000 samples, 3600.0s
   🔄 Resampling: 48000.0

In [6]:
!pip install -r requirements.txt

  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.59.0-cp311-cp311-win_amd64.whl.metadata (110 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
  Using cached kiwisolver-1.4.9-cp311-cp311-win_amd64.whl.metadata (6.4 kB)
  Using cached matplotlib-3.10.5-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached pandas-2.3.1-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-win_amd64.whl.metadata (2.1 kB)
  Using cached pyzmq-27.0.1-cp311-cp311-win_amd64.whl.metadata (6.0 kB)
  Using cached regex-2025.7.34-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  U

ERROR: Could not find a version that satisfies the requirement torch==2.8.0+cu128 (from versions: 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0)
ERROR: No matching distribution found for torch==2.8.0+cu128
